# High Availability, SPOF & Quorum

## 🧠 Mental Models

> **High Availability = the fire drill analogy. Most days nothing happens. But when the
> fire alarm rings (server failure), the system keeps running because you rehearsed,
> have backup routes, and no single locked door (SPOF) blocks everyone.**

> **SPOF = the single load-bearing pillar in a building. If that pillar fails, the
> whole building collapses. HA design = identifying and duplicating every such pillar.**

> **Quorum = a board vote. Decisions require more than half the members to agree.
> If 3 of 5 board members show up, they can vote. If only 2 show up, no decision
> can be made — which prevents split-brain (two groups making contradictory decisions).**


---
## High Availability (HA)

### What HA means in numbers

```
Availability = (Uptime) / (Uptime + Downtime)

99%     ("two nines")   = 87.6 hours/year downtime   ← unacceptable for production
99.9%   ("three nines") = 8.76 hours/year            ← acceptable for some
99.99%  ("four nines")  = 52.6 minutes/year          ← e-commerce target
99.999% ("five nines")  = 5.26 minutes/year          ← financial systems, telecoms

Achieving 99.99% requires:
  - Redundant servers (no SPOF)
  - Automated failover (human can't react in 52 minutes/year!)
  - Multi-AZ or multi-region deployment
  - Chaos engineering (test failure modes regularly)
```

### ❌ What breaks WITHOUT HA design

```
ShopFlow architecture (pre-HA):
  Single app server → All traffic goes here. Crash = total outage.
  Single MySQL DB   → No replica. Crash = data inaccessible until repair (hours).
  Single Redis      → Session cache gone. All users logged out.
  Single load balancer → If LB crashes, no one can reach any server.

Result: 2 hours of downtime during Black Friday = $2M lost revenue
Lesson: EVERY layer must be redundant.
```

### ✅ HA Architecture Layers

```
DNS (Route53)          → Latency/health-based routing across regions
   ↓
Load Balancers (×2)    → Active-passive or active-active pair; auto-failover
   ↓
App Servers (×N)       → Stateless; can lose any one without impact
   ↓
Cache (Redis Cluster)  → Sentinel (3 nodes) or Cluster (6 nodes min)
   ↓
Database Primary       → Synchronous replica in same AZ
   ↓
Database Replica (×1+) → In different AZ; promotes to primary on failure
   ↓
Object Storage (S3)    → Already multi-AZ by design
```

### The HA Decision Matrix

| Component | HA Strategy | Failover Time | Data Loss Risk |
|---|---|---|---|
| App servers | Active-active (N+1) | 0s (LB removes unhealthy) | None (stateless) |
| Redis | Sentinel (auto-failover) | 10-30s | None (replica) |
| PostgreSQL | Streaming replica + Patroni | 30-60s | 0 (synchronous) or seconds (async) |
| Message Queue | Kafka replication factor=3 | 0s (partition leader election) | None |
| Object Storage | S3 / GCS | 0s (managed by provider) | None (11 9s durability) |


---
## Single Point of Failure (SPOF)

### Identifying SPOFs — The Dependency Audit

```
For every component, ask: "If this fails, what else fails?"

Primary DB → App can't write → Orders fail         [SPOF if no replica]
Load Balancer → App unreachable                     [SPOF if single LB]
DNS Provider → App unreachable globally             [SPOF if only one NS]
Payment API Key → Payments fail                     [SPOF if not rotatable]
VPN to on-prem DB → Entire service fails            [SPOF if not redundant]
Shared secrets file → All services crash on restart [SPOF if not in Vault/KMS]
```

### The "Two is One, One is None" Principle

Any component that exists as exactly one is effectively a SPOF. HA design requires
at minimum TWO of every critical component, with automatic failover.

### 🌍 Real SPOF Incidents

| Company | SPOF | Incident | Impact |
|---|---|---|---|
| GitLab (2017) | Single DB, human error | DBA accidentally deleted production DB | 6h outage, partial data loss |
| Cloudflare (2019) | Single WAF rule deployment | Bad regex caused CPU spiral | 27-minute global outage |
| AWS US-East-1 (2021) | Kinesis → downstream chain | Cascading failure across services | Many AWS services down hours |
| Facebook (2021) | BGP config + DNS SPOF | All DCs unreachable | 6h global outage, $6B market cap loss |


In [ ]:
import time, threading, random
from enum import Enum

class State(Enum):
    HEALTHY = "healthy"
    FAILED  = "failed"
    DEGRADED = "degraded"

class Component:
    def __init__(self, name: str, replicas: int = 1):
        self.name     = name
        self.replicas = replicas
        self._failed  = set()

    @property
    def is_spof(self) -> bool:
        return self.replicas == 1

    def fail(self, replica_id: int = 0) -> State:
        self._failed.add(replica_id)
        alive = self.replicas - len(self._failed)
        if alive == 0: return State.FAILED
        if alive < self.replicas: return State.DEGRADED
        return State.HEALTHY

    def recover(self, replica_id: int = 0):
        self._failed.discard(replica_id)

    @property
    def state(self) -> State:
        alive = self.replicas - len(self._failed)
        if alive == 0: return State.FAILED
        if alive < self.replicas: return State.DEGRADED
        return State.HEALTHY

class SystemArchitecture:
    def __init__(self, name: str):
        self.name = name
        self.components: dict[str, Component] = {}

    def add(self, name: str, replicas: int) -> "SystemArchitecture":
        self.components[name] = Component(name, replicas)
        return self

    def audit_spofs(self):
        print(f"
=== SPOF Audit: {self.name} ===")
        for name, comp in self.components.items():
            spof_mark = " ⚠️  SPOF!" if comp.is_spof else " ✓"
            print(f"  {name:25s} replicas={comp.replicas}{spof_mark}")

    def simulate_failure(self, component: str, replica_id: int = 0):
        comp  = self.components[component]
        state = comp.fail(replica_id)
        print(f"
[FAILURE] {component} replica-{replica_id} failed → system state: {state.value}")
        if state == State.FAILED:
            print(f"  ❌ OUTAGE: {component} has NO healthy replicas → system DOWN")
        elif state == State.DEGRADED:
            alive = comp.replicas - len(comp._failed)
            print(f"  ⚠️  DEGRADED: {alive}/{comp.replicas} replicas alive → service continues")

# ── Pre-HA architecture (many SPOFs) ────────────────────────────────────────
print("=== Pre-HA Architecture ===")
pre_ha = (SystemArchitecture("ShopFlow v1")
    .add("Load Balancer",  replicas=1)
    .add("App Servers",    replicas=2)
    .add("Redis Cache",    replicas=1)
    .add("MySQL Primary",  replicas=1)
    .add("MySQL Replica",  replicas=0))  # no replica!
pre_ha.audit_spofs()
pre_ha.simulate_failure("MySQL Primary")

# ── Post-HA architecture (no SPOFs) ─────────────────────────────────────────
print("
=== Post-HA Architecture ===")
post_ha = (SystemArchitecture("ShopFlow v2")
    .add("Load Balancers",  replicas=2)
    .add("App Servers",     replicas=4)
    .add("Redis Sentinel",  replicas=3)   # 3-node Sentinel: 1 master + 2 replicas
    .add("DB Primary",      replicas=1)
    .add("DB Replicas",     replicas=2))  # 2 read replicas + auto-promote
post_ha.audit_spofs()
post_ha.simulate_failure("DB Primary")   # failover, not outage


---
## Quorum

### WHY Quorum Exists — The Split-Brain Problem

```
Scenario: 2 database nodes, network partition between them.
Node A thinks Node B is dead → A promotes itself to primary
Node B thinks Node A is dead → B promotes itself to primary
Both nodes now accept writes → DATA DIVERGENCE

After network heals: which data is correct?
→ You have two conflicting histories. This is split-brain.
→ Fix: require a QUORUM before accepting writes.
```

### WHAT Quorum Is

```
Quorum = ⌊N/2⌋ + 1  (majority of N nodes)

N=3 nodes → quorum = 2   (need any 2 nodes to agree)
N=5 nodes → quorum = 3   (need any 3 nodes to agree)
N=7 nodes → quorum = 4

With quorum:
  3-node cluster, 1 node partitioned:
    - The 2-node majority can still form quorum → continue serving
    - The 1 isolated node can NOT form quorum → refuses to write
    → Split-brain impossible! Only one side can write.
```

### Read Quorum + Write Quorum (Cassandra style)

```
N = replication factor (copies of data)
W = write quorum (how many must confirm a write)
R = read quorum (how many must agree on a read)

Rule: R + W > N  (quorum condition — reads always see latest write)

N=3, W=2, R=2 → R+W=4 > 3 ✓  (strong consistency)
N=3, W=1, R=1 → R+W=2 ≯ 3 ✗  (eventual consistency, fast, may read stale)
N=3, W=3, R=1 → R+W=4 > 3 ✓  (every node must write, single-node reads)
```

### 🌍 Where Quorum is Used

| System | Quorum usage |
|---|---|
| Apache ZooKeeper | Write majority required; 3 or 5 node clusters |
| etcd (Kubernetes) | Raft consensus; quorum of 3 or 5 |
| Apache Cassandra | Configurable W/R quorum per query |
| MongoDB Replica Set | Write to majority for durability |
| DynamoDB | Quorum-based replication internally |


In [ ]:
import threading, time, random
from enum import Enum

class NodeRole(Enum):
    FOLLOWER  = "follower"
    CANDIDATE = "candidate"
    LEADER    = "leader"

class QuorumCluster:
    '''
    Simplified quorum simulation.
    Demonstrates why N=3 survives 1 failure but N=2 causes split-brain.
    '''
    def __init__(self, n_nodes: int):
        self.n        = n_nodes
        self.quorum   = n_nodes // 2 + 1
        self._alive   = set(range(n_nodes))
        self._data    = {}   # key → value (replicated)
        self._version = 0

    @property
    def alive_count(self) -> int:
        return len(self._alive)

    @property
    def can_write(self) -> bool:
        '''Can we achieve write quorum?'''
        return self.alive_count >= self.quorum

    def fail_node(self, node_id: int):
        self._alive.discard(node_id)
        print(f"  [Cluster] Node {node_id} FAILED. Alive: {sorted(self._alive)} "
              f"({self.alive_count}/{self.n}) — quorum={'✓' if self.can_write else '✗ LOST'}")

    def recover_node(self, node_id: int):
        self._alive.add(node_id)
        print(f"  [Cluster] Node {node_id} RECOVERED. Alive: {sorted(self._alive)}")

    def write(self, key: str, value) -> bool:
        if not self.can_write:
            print(f"  [Write REJECTED] No quorum ({self.alive_count}/{self.quorum} needed). "
                  f"key={key!r} — split-brain prevention!")
            return False
        self._version += 1
        self._data[key] = (value, self._version)
        print(f"  [Write OK] key={key!r}={value!r} v{self._version} "
              f"(confirmed by {self.alive_count}/{self.n} nodes)")
        return True

# ── Demonstrate quorum with N=3 ──────────────────────────────────────────────
print("=== Quorum Demo: N=3, quorum=2 ===")
cluster = QuorumCluster(n_nodes=3)
cluster.write("leader", "node-0")

print("
Simulate 1 node failure:")
cluster.fail_node(1)
print("Can still write? (2 nodes alive, quorum=2):")
cluster.write("status", "degraded-but-alive")

print("
Simulate 2nd node failure:")
cluster.fail_node(2)
print("Can still write? (1 node alive, quorum=2):")
cluster.write("status", "this-would-cause-split-brain")  # should be rejected

print("
Node 2 recovers:")
cluster.recover_node(2)
cluster.write("status", "recovering")

# ── WHY N=2 is dangerous ─────────────────────────────────────────────────────
print("
=== WHY NOT N=2 (no quorum possible without all nodes) ===")
cluster2 = QuorumCluster(n_nodes=2)
cluster2.write("data", "initial")
cluster2.fail_node(0)
print("With N=2, losing 1 node = losing quorum:")
cluster2.write("data", "update")   # rejected — prevents split-brain, but also prevents all writes!
# → That's why N=2 is useless: you can't make progress if any node fails
# → Use N=3 as minimum (survives 1 failure, can still write with 2/3)
print("
KEY INSIGHT: Always use odd N (3, 5, 7) — even N provides no extra fault tolerance.")
